# Pipeline Unificado: Construccion, Curaduria y Enriquecimiento del Corpus de Citas Cientificas

## Descripcion General del Cuaderno
Este cuaderno ejecuta de manera lineal, estructurada y reproducible todo el proceso de construccion del conjunto de datos de citas cientificas para la clasificacion de funciones retoricas (9 categorias) y la extraccion densa de pasajes documentales (*Top-3 Chunks*).

A diferencia de las etapas exploratorias previas, este flujo unifica de forma directa:
1. **Ingesta Multi-Fuente Genuina:** Carga y estandarizacion de fuentes primarias (SciCite y SciTail), enriquecidas con consultas dirigidas en Semantic Scholar (surveys, articulos seminales, herramientas y comparativas empiricas) y ACL-ARC.
2. **Evidencia Documental Real:** Recuperacion sistematica de titulos y resumenes (*abstracts*) de los articulos citados mediante consultas por lotes a Semantic Scholar API.
3. **Clasificacion Asistida con Llama y Rescate Secundario:** Etiquetado en GPU mediante un modelo de lenguaje abierto (Llama cuantizado en 4 bits) y reasignacion de citas candidatas del excedente mayoritario (*Basis* y *Background*) hacia clases deficitarias (*Further_Reading*, *Modification_Improvement* y *Application*), sin recurrir a texto sintetico.
4. **Submuestreo Inteligente con Priorizacion de Evidencia:** Seleccion de hasta 2.000 instancias por categoria priorizando aquellas que cuentan con soporte textual verificable del trabajo citado.
5. **Aislamiento Documental Estricto (*No-Leakage*):** Particion estratificada agrupada por documento citante (`citing_paper_id`) con cero solapamiento entre Entrenamiento (70%), Validacion (15%) y Prueba Humana (15%).
6. **Recuperacion Densa de Pasajes (*Top-3 Chunks*):** Segmentacion oracional y calculo de similitud semantica del coseno con `SentenceTransformer`, garantizando rigor cientifico: si un articulo no tiene texto disponible, se preserva vacio (`NaN`) sin autoreferencias circulares.
7. **Esquema Definitivo Limpio (23 Columnas):** Eliminacion de campos auxiliares redundantes y persistencia de las particiones validadas en Google Drive.


## 1. Configuracion del Entorno, Google Drive y Credenciales

### Explicacion de la fase:
Para ejecutar este flujo necesitamos conectar nuestro almacenamiento permanente en Google Drive (carpeta `Dataset_final`) para guardar copias de seguridad de cada etapa (*checkpoints*). De esta forma, si la conexion se interrumpe, el cuaderno reanuda desde el ultimo punto sin repetir calculos largos.

Tambien verificamos que Colab disponga de aceleracion por GPU (T4, L4 o A100) para cargar Llama en 4 bits de forma rapida y eficiente.


In [ ]:
# 1.1 Instalacion de dependencias requeridas
!pip install -q transformers accelerate bitsandbytes sentence-transformers datasets aiohttp huggingface_hub pydantic scikit-learn tqdm
!pip install -q transformers accelerate bitsandbytes sentence-transformers "datasets<3.0.0" aiohttp huggingface_hub pydantic scikit-learn tqdm

# 1.2 Verificacion de GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Este cuaderno requiere un entorno con GPU. En el menu superior selecciona: Entorno de ejecucion -> Cambiar tipo de entorno -> GPU.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"[GPU] Conectado exitosamente a: {gpu_name} ({gpu_mem:.2f} GB VRAM)")

# 1.3 Modulos base del sistema
import os
import re
import json
import time
import io
import tarfile
import shutil
import urllib.parse
import urllib.request
import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive, userdata
from huggingface_hub import login

# 1.4 Conexion con Google Drive y definicion de carpeta de trabajo
drive.mount('/content/drive', force_remount=False)
DRIVE_DIR = '/content/drive/MyDrive/Dataset_final'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"[Drive] Carpeta de persistencia activa: {DRIVE_DIR}")

# 1.5 Carga segura de tokens desde Colab Secrets
HF_TOKEN = None
S2_API_KEY = None

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        login(token=str(HF_TOKEN).strip().replace('\r', '').replace('\n', ''))
        print("[Auth] Hugging Face autenticado correctamente.")
except Exception as e:
    print(f"[Auth] HF_TOKEN no configurado en Secrets: {e}")

try:
    S2_API_KEY = userdata.get('S2_API_KEY')
    if S2_API_KEY:
        S2_API_KEY = str(S2_API_KEY).strip().replace('\r', '').replace('\n', '')
        print("[Auth] S2_API_KEY detectada para Semantic Scholar API.")
except Exception as e:
    print(f"[Auth] S2_API_KEY no configurada (se usara acceso publico con tasa limitada): {e}")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.6.1 which is incompatible.
[GPU] Conectado exitosamente a: NVIDIA L4 (23.66 GB VRAM)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Drive] Carpeta de persistencia activa: /content/drive/MyDrive/Dataset_final
[Auth] Hugging Face autenticado correctamente.
[Auth] S2_API_KEY detectada para Semantic Scholar API.


## 2. Ingesta Multi-Fuente y Recuperacion de Metadatos

### Explicacion de la fase:
En la literatura cientifica, las citas no se distribuyen de manera uniforme: la mayoria de los autores citan trabajos para dar contexto general (*Background*) o como base conceptual (*Basis*), mientras que funciones como senalar lecturas adicionales (*Further_Reading*), proponer adaptaciones de metodos (*Modification_Improvement*) o usar herramientas practicas (*Application*) son mucho mas escasas.

Para corregir este desbalance desde el origen sin inventar datos sinteticos, combinamos cuatro fuentes complementarias:
1. **SciCite y SciTail:** Citas y premisas cientificas estandarizadas de libre acceso.
2. **Semantic Scholar (API Graph):** Consultas disenadas tematicamente para encontrar articulos de revision de literatura (*surveys*), bibliotecas de software cientifico (*toolkits*), articulos seminales muy citados y estudios comparativos de rendimiento.
3. **ACL-ARC:** Citas de alta calidad procedentes de la antologia de linguistica computacional.
4. **Recuperacion en Lote de Titulos y Abstracts (`/paper/batch`):** Para aquellas citas que solo tenian el identificador del paper citado, consultamos Semantic Scholar para traer su titulo oficial y su resumen real.


In [ ]:
# 2.1 Descarga y preparacion de SciCite y SciTail
import io
import tarfile
import urllib.request
import shutil
from datasets import load_dataset

checkpoint_primarias = os.path.join(DRIVE_DIR, "checkpoint_fuentes_primarias_scicite_scitail.csv")
checkpoint_previo = "/content/drive/MyDrive/Citas_NLP/checkpoint_fuentes_primarias_scicite_scitail.csv"

# Reutilizar checkpoint previo si ya existia en Citas_NLP
if not os.path.exists(checkpoint_primarias) and os.path.exists(checkpoint_previo):
    print("[Optimizacion] Reutilizando checkpoint previo de fuentes primarias desde Citas_NLP...")
    shutil.copy2(checkpoint_previo, checkpoint_primarias)

if os.path.exists(checkpoint_primarias):
    df_primarias = pd.read_csv(checkpoint_primarias)
    print(f"[Carga] Fuentes primarias recuperadas de Drive: {len(df_primarias):,} citas.")
else:
    print("[Ingesta] Descargando SciCite y SciTail desde Hugging Face...")
    ds_scicite = load_dataset("allenai/scicite", split="train")
    ds_scitail = load_dataset("allenai/scitail", "snli_format", split="train")
    print("[Ingesta] Descargando SciCite desde repositorio oficial de AllenAI (S3)...")
    scicite_url = "https://s3-us-west-2.amazonaws.com/ai2-s2-research/scicite/scicite.tar.gz"
    req = urllib.request.Request(scicite_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp:
        tar_bytes = io.BytesIO(resp.read())

    with tarfile.open(fileobj=tar_bytes, mode="r:gz") as tar:
        f_train = tar.extractfile("scicite/train.jsonl")
        df_scicite_raw = pd.read_json(io.BytesIO(f_train.read()), lines=True)

    registros = []

    # SciCite: Mapeo de etiquetas
    mapa_scicite = {
        'background': 'Background',
        'method': 'Basis',
        'result': 'Evidence'
    }
    for row in tqdm(ds_scicite, desc="Procesando SciCite"):
        lbl_raw = row.get('label')
        lbl_str = ds_scicite.features['label'].int2str(lbl_raw) if isinstance(lbl_raw, int) else str(lbl_raw)
    for _, row in tqdm(df_scicite_raw.iterrows(), total=len(df_scicite_raw), desc="Procesando SciCite"):
        lbl_str = str(row.get('label', '')).strip().lower()
        registros.append({
            'citation_context': str(row.get('string', '')).strip(),
            'rhetorical_section': str(row.get('sectionName', 'Unknown')).strip(),
            'citing_paper_id': str(row.get('citingPaperId', '')).strip(),
            'cited_paper_id': str(row.get('citedPaperId', '')).strip(),
            'source_dataset': 'scicite',
            'label': mapa_scicite.get(lbl_str.lower(), 'Background'),
            'label': mapa_scicite.get(lbl_str, 'Background'),
            'sentence2': np.nan,
            'cited_paper_title': np.nan,
            'cited_paper_abstract': np.nan
        })

    # SciTail: Premisas de soporte cientifico
    print("[Ingesta] Descargando SciTail...")
    try:
        ds_scitail = load_dataset("allenai/scitail", "snli_format", split="train")
    except Exception:
        ds_scitail = load_dataset("parquet", data_files="https://huggingface.co/api/datasets/allenai/scitail/parquet/snli_format/train/0.parquet", split="train")

    for row in tqdm(ds_scitail, desc="Procesando SciTail"):
        lbl = str(row.get('gold_label', '')).lower()
        cat = 'Basis' if lbl == 'entails' else 'Background'
        registros.append({
            'citation_context': str(row.get('sentence1', '')).strip(),
            'rhetorical_section': 'Scientific Premise',
            'citing_paper_id': f"scitail_{row.get('pairID', '')}_citing",
            'cited_paper_id': f"scitail_{row.get('pairID', '')}_cited",
            'source_dataset': 'scitail',
            'label': cat,
            'sentence2': str(row.get('sentence2', '')).strip(),
            'cited_paper_title': np.nan,
            'cited_paper_abstract': np.nan
        })

    df_primarias = pd.DataFrame(registros)
    df_primarias.to_csv(checkpoint_primarias, index=False)
    print(f"[Guardado] Checkpoint de primarias guardado: {len(df_primarias):,} registros.")


[Carga] Fuentes primarias recuperadas de Drive: 36,928 citas.


In [ ]:
# 2.2 Consultas dirigidas a Semantic Scholar y recuperacion de metadatos
checkpoint_s2 = os.path.join(DRIVE_DIR, "checkpoint_s2_ingesta_unificada.csv")

headers_s2 = {}
if S2_API_KEY:
    headers_s2['x-api-key'] = S2_API_KEY

def peticion_s2_segura(url, max_intentos=3):
    for intento in range(max_intentos):
        time.sleep(1.15)  # Respeto riguroso de la tasa de 1 req/s
        try:
            r = requests.get(url, headers=headers_s2, timeout=20)
            if r.status_code == 200:
                return r
            elif r.status_code == 429:
                time.sleep(6.0 * (intento + 1))
            else:
                time.sleep(1.5)
        except Exception:
            time.sleep(2.0)
    return None

if os.path.exists(checkpoint_s2):
    df_s2_recolectado = pd.read_csv(checkpoint_s2)
    print(f"[Carga] Citas de Semantic Scholar recuperadas de Drive: {len(df_s2_recolectado):,} citas.")
else:
    print("[Ingesta] Ejecutando consultas tematicas dirigidas en Semantic Scholar API...")

    # Consultas para atraer articulos de revision, extensiones y comparativas
    QUERIES_DIRIGIDAS = [
        # Further_Reading (Surveys y tutoriales)
        "comprehensive survey deep learning literature review",
        "a survey on natural language processing review",
        "recent advances large language models survey",
        "systematic review computer vision methods tutorial",
        # Modification_Improvement (Adaptaciones y extensiones)
        "we extend the architecture modified framework deep learning",
        "an improved variant adapted convolutional neural networks",
        "modified transformer architecture extending prior model",
        # Application (Herramientas y librerias)
        "open source toolkit library implementation python framework",
        "applied machine learning software package implementation",
        # Comparison (Estudios empiricos y benchmarks)
        "empirical evaluation benchmark comparison outperforms baseline",
        "comparative study evaluating performance state of the art",
        # Articulos seminales de alta citacion
        "Attention Is All You Need",
        "BERT: Pre-training of Deep Bidirectional Transformers",
        "Deep Residual Learning for Image Recognition"
    ]

    citas_s2 = []
    contextos_vistos = set(df_primarias['citation_context'].astype(str))

    for q in tqdm(QUERIES_DIRIGIDAS, desc="Buscando en S2"):
        url_search = f"https://api.semanticscholar.org/graph/v1/paper/search?query={urllib.parse.quote(q)}&limit=10&fields=paperId,title,abstract"
        r_s = peticion_s2_segura(url_search)
        if not r_s:
            continue

        papers = r_s.json().get('data', [])
        for p in papers:
            pid = p.get('paperId')
            p_title = p.get('title')
            p_abstract = p.get('abstract')

            if not pid:
                continue

            # Consultar citas entrantes de este articulo
            url_cit = f"https://api.semanticscholar.org/graph/v1/paper/{pid}/citations?fields=contexts,intents,citingPaper.paperId,citingPaper.title&limit=100"
            r_cit = peticion_s2_segura(url_cit)
            if not r_cit:
                continue

            data_cit = r_cit.json().get('data', [])
            for c_entry in data_cit:
                c_citing = c_entry.get('citingPaper', {})
                citing_id = c_citing.get('paperId')
                contexts = c_entry.get('contexts', [])

                for ctx in contexts:
                    ctx_limpio = str(ctx).strip()
                    if len(ctx_limpio.split()) >= 6 and ctx_limpio not in contextos_vistos:
                        contextos_vistos.add(ctx_limpio)
                        citas_s2.append({
                            'citation_context': ctx_limpio,
                            'rhetorical_section': 'Unknown',
                            'citing_paper_id': citing_id if citing_id else f"s2_citing_{len(citas_s2)}",
                            'cited_paper_id': pid,
                            'source_dataset': 'Semantic_Scholar_Consolidado',
                            'label': 'Background',  # Etiqueta temporal que pasara a evaluacion
                            'sentence2': np.nan,
                            'cited_paper_title': p_title,
                            'cited_paper_abstract': p_abstract
                        })

    df_s2_recolectado = pd.DataFrame(citas_s2)
    df_s2_recolectado.to_csv(checkpoint_s2, index=False)
    print(f"[Guardado] Citas recolectadas de S2 guardadas: {len(df_s2_recolectado):,} registros.")


[Carga] Citas de Semantic Scholar recuperadas de Drive: 1,551 citas.


In [ ]:
# 2.3 Recuperacion en lote (S2 Batch) de Abstracts y Titulos faltantes
checkpoint_unificado_crudo = os.path.join(DRIVE_DIR, "checkpoint_candidatos_totales_con_metadatos.csv")

if os.path.exists(checkpoint_unificado_crudo):
    df_candidatos = pd.read_csv(checkpoint_unificado_crudo)
    print(f"[Carga] Candidatos totales con metadatos cargados: {len(df_candidatos):,} filas.")
else:
    df_candidatos = pd.concat([df_primarias, df_s2_recolectado], ignore_index=True)

    # Identificar articulos citados que tienen hash pero no abstract
    def es_hash_valido(v):
        s = str(v).strip()
        return len(s) == 40 and not s.startswith('scitail')

    candidatos_ids = df_candidatos[
        df_candidatos['cited_paper_id'].apply(es_hash_valido) &
        (df_candidatos['cited_paper_abstract'].isna() | (df_candidatos['cited_paper_abstract'].astype(str).str.len() < 25))
    ]['cited_paper_id'].unique().tolist()

    print(f"[S2 Batch] Articulos citados unicos a consultar para recuperar abstract/titulo: {len(candidatos_ids):,}")

    catalogo_metadatos = {}
    tamano_lote = 100

    for i in tqdm(range(0, len(candidatos_ids), tamano_lote), desc="Descargando Abstracts en Lote"):
        lote_ids = candidatos_ids[i:i + tamano_lote]
        url_batch = "https://api.semanticscholar.org/graph/v1/paper/batch?fields=paperId,title,abstract"

        for intento in range(3):
            time.sleep(1.2)
            try:
                r = requests.post(url_batch, json={"ids": lote_ids}, headers=headers_s2, timeout=25)
                if r.status_code == 200:
                    for item in r.json():
                        if item and item.get('paperId'):
                            catalogo_metadatos[item['paperId']] = {
                                'title': item.get('title'),
                                'abstract': item.get('abstract')
                            }
                    break
                elif r.status_code == 429:
                    time.sleep(5.0 * (intento + 1))
            except Exception:
                time.sleep(2.0)

    # Inyectar metadatos recuperados
    for idx in df_candidatos.index:
        cid = str(df_candidatos.at[idx, 'cited_paper_id']).strip()
        if cid in catalogo_metadatos:
            meta = catalogo_metadatos[cid]
            if meta['abstract'] and (pd.isna(df_candidatos.at[idx, 'cited_paper_abstract']) or len(str(df_candidatos.at[idx, 'cited_paper_abstract'])) < 25):
                df_candidatos.at[idx, 'cited_paper_abstract'] = meta['abstract']
            if meta['title'] and (pd.isna(df_candidatos.at[idx, 'cited_paper_title']) or len(str(df_candidatos.at[idx, 'cited_paper_title'])) < 5):
                df_candidatos.at[idx, 'cited_paper_title'] = meta['title']

    df_candidatos.to_csv(checkpoint_unificado_crudo, index=False)
    print(f"[Guardado] Candidatos consolidados y enriquecidos: {len(df_candidatos):,} filas.")


[Carga] Candidatos totales con metadatos cargados: 38,479 filas.


## 3. Filtrado de Calidad y Deteccion de Marcadores Lexicos

### Explicacion de la fase:
No todas las oraciones extraidas de articulos cientificos son adecuadas para entrenar un modelo. Citas de 2 o 3 palabras (como *"see [1]"* aislado) no contienen contexto retorico suficiente.

En esta fase aplicamos un filtro de purificacion para:
1. Eliminar oraciones demasiado cortas (menos de 6 palabras) o que son artefactos de encabezados.
2. Identificar mediante expresiones regulares las citas que contienen marcadores discursivos tipicos de funciones escasas (ej. *"we extend"*, *"we employ"*, *"outperforms"*, *"refer to for a survey"*), para canalizarlas de manera prioritaria a la evaluacion de Llama.


In [ ]:
# 3.1 Limpieza y normalizacion basica de citas
def limpiar_cita(texto):
    if pd.isna(texto):
        return ""
    t = str(texto).strip()
    t = re.sub(r'\s+', ' ', t)
    return t

df_candidatos['citation_context'] = df_candidatos['citation_context'].apply(limpiar_cita)
df_candidatos = df_candidatos[df_candidatos['citation_context'].apply(lambda x: len(x.split()) >= 6)].reset_index(drop=True)

# 3.2 Marcadores lexicos para pre-clasificacion rapida
PATRONES_LEXICOS = {
    "Further_Reading": re.compile(r'\b(?:see also|refer to|for further details|for a survey|for a review|tutorial|comprehensive overview|detailed survey in)\b', re.IGNORECASE),
    "Modification_Improvement": re.compile(r'\b(?:extend|extended|extending|modified|modifying|adapting|adapted from|improved upon|variant of|build upon|enhance)\b', re.IGNORECASE),
    "Application": re.compile(r'\b(?:we use|we employ|applied|implemented using|utilizing|toolkit|library|package|software|framework)\b', re.IGNORECASE),
    "Evidence": re.compile(r'\b(?:consistent with|confirming|supports|demonstrated by|as shown in|corroborates|empirical evidence)\b', re.IGNORECASE),
    "Comparison": re.compile(r'\b(?:in contrast|outperforms|compared to|unlike|whereas|better than|superior to|differs from)\b', re.IGNORECASE)
}

print(f"[Calidad] Total de citas validas tras depuracion de texto: {len(df_candidatos):,}")


[Calidad] Total de citas validas tras depuracion de texto: 37,271


## 4. Clasificacion con Llama en GPU y Rescate Probabilistico

### Explicacion de la fase:
¿Por que usamos Llama y por que aplicamos un *rescate probabilistico*?
* **Precision contextual de Llama:** Los modelos tradicionales por palabras clave no entienden el matiz de la oracion. Llama (ejecutado en GPU con cuantizacion a 4 bits) lee la cita y determina la verdadera intencion retorica del autor.
* **El mecanismo de rescate:** En cualquier base de datos de citas, mas del 60% se etiquetan por defecto como contexto general (*Background*) o base teorica (*Basis*). Sin embargo, al inspeccionar el texto, muchas de esas citas en realidad describen que los autores *modificaron una arquitectura*, *utilizaron una libreria de software* o *recomiendan leer un tutorial*.
* El rescate consiste en tomar el excedente de las clases saturadas y pedirle a Llama que identifique si pertenecen a las categorias deficitarias. De esta forma alcanzamos el volumen necesario **utilizando unicamente citas reales de cientificos, con cero datos artificiales o inventados**.


In [ ]:
# ====================================================================================================
# 4.1 Clasificación Inicial con Llama 3.2 (3B Instruct) y Obtención de Vector de Scores
# ====================================================================================================
import os
import re
import json
import shutil
import time
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
checkpoint_clasificacion = os.path.join(DRIVE_DIR, "checkpoint_citas_clasificadas_llama.csv")
checkpoint_previo_llama  = "/content/drive/MyDrive/Citas_NLP/checkpoint_citas_clasificadas_llama.csv"
# 1. Detección de herencia inteligente de checkpoint existente en Drive
necesita_procesar = True
if os.path.exists(checkpoint_clasificacion):
    try:
        df_clas = pd.read_csv(checkpoint_clasificacion)
        if df_clas['label'].nunique() >= 9 and 'scores_json' in df_clas.columns:
            print(f"[Carga] Checkpoint validado en Drive ({len(df_clas):,} citas, {df_clas['label'].nunique()} clases con scores).")
            necesita_procesar = False
        else:
            print(f"[Aviso] El checkpoint en {DRIVE_DIR} requiere regeneración con scores.")
    except Exception as e:
        print(f"[Aviso] Error al leer checkpoint existente: {e}")
if necesita_procesar and os.path.exists(checkpoint_previo_llama):
    try:
        df_previo = pd.read_csv(checkpoint_previo_llama)
        if df_previo['label'].nunique() >= 9 and 'scores_json' in df_previo.columns:
            print(f"[Optimización] Heredando checkpoint maestro desde Citas_NLP ({len(df_previo):,} citas)...")
            shutil.copy2(checkpoint_previo_llama, checkpoint_clasificacion)
            df_clas = df_previo.copy()
            necesita_procesar = False
    except Exception as e:
        print(f"[Aviso] No se pudo copiar desde Citas_NLP: {e}")
# 2. Inicialización de Llama 3.2 y clasificación inicial
if necesita_procesar:
    MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
    print(f"[Modelo] Inicializando {MODEL_ID} en cuantización de 4 bits...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN
    )
    model.eval()
    print("[Modelo] Llama 3.2 cargado exitosamente en GPU.")
    CLASES_VALIDAS = [
        "Background", "Gap", "Basis", "Comparison",
        "Application", "Modification_Improvement", "Evidence",
        "Identification_of_the_Originator", "Further_Reading"
    ]
    SYSTEM_PROMPT = """You are an expert in Computational Linguistics and NLP specialized in scientific citation analysis.
Classify the citation intent into exactly ONE of the following 9 official categories defined in the project specification:
1. Background:
  - Definition: Citations used to provide context, summarize the general background of a research topic, or trace the history of a field or idea. Lay the foundation for understanding the focal study.
  - Criteria: Reviews existing literature; cites review papers or previous studies depicting the current state of the field; highlights prevailing findings; traces historical development.
2. Gap:
  - Definition: Citations that help identify research gaps or unexplored areas, justifying the author's choice of research topic.
  - Criteria: Highlights what has or has not been done; identifies gaps in knowledge or areas for further study; justifies relevance by contrasting current research with existing work.
3. Basis:
  - Definition: Citations that provide a foundation for the current research, operating at a macro level (broad influence) or micro level (specific key contributions).
  - Criteria: Serves as the intellectual starting point; significantly shapes the research idea or hypothesis; the focal research builds upon, continues, or expands on the cited work.
4. Comparison:
  - Definition: Citations used to compare the current work with cited studies or to draw comparisons between cited studies.
  - Criteria: Highlights similarities or differences; compares methodologies, findings, algorithms, data, or theoretical concepts; points out advantages over previous studies.
5. Application:
  - Definition: Citations that directly employ a method, technique, or tool from the cited work without modification.
  - Criteria: Utilizes existing methods, algorithms, instruments, or data; employs the cited work as a practical tool (equations, software, analysis methods) for calculation or experimentation.
6. Modification_Improvement:
  - Definition: Citations in which methods or tools from the cited work are adapted, expanded, or modified for the current research.
  - Criteria: The cited method is adapted, improved, or extended to fit new experimental conditions or research objectives; enhances precision, efficiency, or scope.
7. Evidence:
  - Definition: Citations used to support claims, hypotheses, or findings in the current research.
  - Criteria: Supports arguments, hypotheses, or factual statements; justifies research design or experimental procedures; substantiates findings (especially in discussion); provides evidence to mitigate limitations.
8. Identification_of_the_Originator:
  - Definition: Citations used to acknowledge the original source of an idea, concept, method, or theory.
  - Criteria: Identifies the original publication where a key idea or method was first introduced; acknowledges pioneers; gives credit demonstrating intellectual indebtedness.
9. Further_Reading:
  - Definition: Citations that direct readers to additional or supplementary literature for more detailed information or context.
  - Criteria: Alerts readers to external, new, or supplementary sources; provides details on data, methods, or background not fully covered in the current text.
Instructions:
- Analyze the rhetorical section, the citation context, and the cited paper details (if available).
- Respond ONLY with a valid compact JSON containing:
  1. 'predicted_label': The dominant category name among the 9 official classes.
  2. 'scores': A dictionary containing the estimated probability or confidence score (float between 0.0 and 1.0) for each of the 9 categories.
Example format:
{"predicted_label": "<dominant_category>", "scores": {"Background": 0.0, "Gap": 0.0, "Basis": 0.0, "Comparison": 0.0, "Application": 0.0, "Modification_Improvement": 0.0, "Evidence": 0.0, "Identification_of_the_Originator": 0.0, "Further_Reading": 0.0}}"""
    def clasificar_lote(batch_items):
        prompts = []
        for item in batch_items:
            ctx = str(item['citation_context']).strip()
            sec = str(item.get('rhetorical_section', 'Scientific Body'))
            tit = str(item.get('cited_paper_title', '')).strip()
            abs_txt = str(item.get('cited_paper_abstract', '')).strip()
            user_content = f"Rhetorical Section: {sec}\nCitation Context: \"{ctx}\""
            if tit and len(tit) > 5 and tit.lower() != 'nan':
                user_content += f"\nCited Paper Title: \"{tit}\""
            if abs_txt and len(abs_txt) > 25 and abs_txt.lower() != 'nan':
                user_content += f"\nCited Paper Abstract: \"{abs_txt[:300]}...\""
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content}
            ]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            prompts.append(prompt)
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1536).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=110,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )
        input_len = inputs.input_ids.shape[1]
        respuestas = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
        resultados = []
        for resp_str in respuestas:
            pred_raw = "Background"
            scores_dict = {cv: 0.0 for cv in CLASES_VALIDAS}
            match = re.search(r'\{.*\}', resp_str, re.DOTALL)
            if match:
                try:
                    data = json.loads(match.group(0))
                    pred_raw = str(data.get('predicted_label', 'Background')).strip()
                    raw_scores = data.get('scores', {})
                    if isinstance(raw_scores, dict):
                        for k, v in raw_scores.items():
                            for cv in CLASES_VALIDAS:
                                if cv.lower() == str(k).strip().lower():
                                    try:
                                        scores_dict[cv] = round(float(v), 4)
                                    except:
                                        scores_dict[cv] = 0.0
                except Exception:
                    pass
            pred_limpio = "Background"
            for cv in CLASES_VALIDAS:
                if cv.lower() == pred_raw.lower() or cv.lower() in pred_raw.lower():
                    pred_limpio = cv
                    break
            resultados.append({
                "label": pred_limpio,
                "predicted_label": pred_limpio,
                "scores_json": json.dumps(scores_dict),
                "etiqueta_rescatada": False
            })
        return resultados
    BATCH_SIZE = 16
    items = df_candidatos.to_dict('records')
    clasificados = []
    print(f"[Inferencia] Clasificando {len(items):,} citas con Llama 3.2 en lotes de {BATCH_SIZE}...")
    for i in tqdm(range(0, len(items), BATCH_SIZE), desc="Clasificación Inicial Llama 3.2"):
        batch = items[i:i + BATCH_SIZE]
        resultados_batch = clasificar_lote(batch)
        for it, res in zip(batch, resultados_batch):
            it['label'] = res['label']
            it['predicted_label'] = res['predicted_label']
            it['scores_json'] = res['scores_json']
            it['etiqueta_rescatada'] = res['etiqueta_rescatada']
            clasificados.append(it)
        if (i // BATCH_SIZE) % 50 == 0 or (i + BATCH_SIZE) >= len(items):
            pd.DataFrame(clasificados).to_csv(checkpoint_clasificacion, index=False)
    df_clas = pd.DataFrame(clasificados)
    df_clas.to_csv(checkpoint_clasificacion, index=False)
    print(f"[Guardado] Clasificación inicial completa persistida: {len(df_clas):,} citas.")
print("\nDISTRIBUCIÓN INICIAL (PREVIO AL RESCATE):")
print(df_clas['predicted_label'].value_counts())

[Aviso] El checkpoint en /content/drive/MyDrive/Dataset_final solo contiene 1 clase(s). Requiere actualizacion.
[Optimizacion] Heredando checkpoint maestro de Llama desde Citas_NLP (41,172 citas, 9 clases)...

DISTRIBUCION ACTUAL POR CLASE RETORICA EN df_clas:
label
Background                          12171
Basis                               12139
Identification_of_the_Originator     6600
Gap                                  3261
Application                          2064
Comparison                           1983
Evidence                             1728
Modification_Improvement              660
Further_Reading                       566
Name: count, dtype: int64


In [ ]:
# ====================================================================================================
# 4.2 Algoritmo de Rescate Probabilístico Secundario sobre Clases Deficitarias
# ====================================================================================================
import json
import pandas as pd
# 1. Configuración de parámetros de rescate
CLASES_CRITICAS = ["Application", "Modification_Improvement", "Further_Reading"]
CLASES_SATURADAS = ["Basis", "Background"]
META_MINIMA = 1200
UMBRAL_PRIMARIO = 0.30
UMBRAL_SECUNDARIO = 0.20
# 2. Conteo de partida
conteo_actual = df_clas['label'].value_counts().to_dict()
print("=" * 80)
print("DISTRIBUCIÓN ANTES DEL RESCATE PROBABILÍSTICO:")
print("=" * 80)
for c, cnt in sorted(conteo_actual.items(), key=lambda x: x[1], reverse=True):
    print(f"  - {c:<35}: {cnt:6,} citas")
# 3. Función de evaluación de rescate por fila
def aplicar_rescate(row):
    pred = row['predicted_label']
    scores_str = row.get('scores_json', '{}')

    try:
        scores = json.loads(scores_str) if isinstance(scores_str, str) else {}
    except:
        scores = {}
    # Solo se rescata si la predicción inicial es una clase saturada
    if pred in CLASES_SATURADAS:
        deficitarias = [c for c in CLASES_CRITICAS if conteo_actual.get(c, 0) < META_MINIMA]
        if deficitarias:
            # Regla 1: Candidatos con Score >= 0.30
            cands_30 = []
            for c in deficitarias:
                val = float(scores.get(c, 0.0))
                if val >= UMBRAL_PRIMARIO:
                    cands_30.append((c, val))
            if cands_30:
                cands_30.sort(key=lambda x: x[1], reverse=True)
                ganador = cands_30[0][0]
                conteo_actual[ganador] = conteo_actual.get(ganador, 0) + 1
                conteo_actual[pred] -= 1
                return pd.Series([ganador, True, f"Regla 1 (Score {cands_30[0][1]:.2f} >= {UMBRAL_PRIMARIO})"])
            # Regla 2: Candidato con mayor afinidad latente >= 0.20
            max_c, max_v = None, 0.0
            for c in deficitarias:
                val = float(scores.get(c, 0.0))
                if val > max_v:
                    max_v, max_c = val, c
            if max_v >= UMBRAL_SECUNDARIO and max_c is not None:
                conteo_actual[max_c] = conteo_actual.get(max_c, 0) + 1
                conteo_actual[pred] -= 1
                return pd.Series([max_c, True, f"Regla 2 (Score {max_v:.2f} >= {UMBRAL_SECUNDARIO})"])
    return pd.Series([pred, False, "Sin rescate (Mantiene etiqueta inicial)"])
# 4. Ejecución del rescate sobre el DataFrame
df_clas[['label', 'etiqueta_rescatada', 'motivo_rescate']] = df_clas.apply(aplicar_rescate, axis=1)
# 5. Persistencia del DataFrame actualizado
checkpoint_clasificacion = os.path.join(DRIVE_DIR, "checkpoint_citas_clasificadas_llama.csv") if 'DRIVE_DIR' in globals() else "checkpoint_citas_clasificadas_llama.csv"
df_clas.to_csv(checkpoint_clasificacion, index=False)
# 6. Reporte comparativo antes vs después
total_rescatadas = df_clas['etiqueta_rescatada'].sum()
print("\n" + "=" * 80)
print(f"AUDITORÍA FINAL TRAS EL RESCATE PROBABILÍSTICO (Total Rescatadas: {total_rescatadas:,}):")
print("=" * 80)
tabla_comparativa = []
for c in sorted(df_clas['label'].unique()):
    antes = (df_clas['predicted_label'] == c).sum()
    despues = (df_clas['label'] == c).sum()
    diff = despues - antes
    tabla_comparativa.append({
        "Categoría": c,
        "Antes del Rescate": antes,
        "Tras el Rescate": despues,
        "Variación Neta": f"+{diff:,}" if diff > 0 else f"{diff:,}"
    })
print(pd.DataFrame(tabla_comparativa).to_string(index=False))
# 7. Ejemplo representativo antes y después del rescate
print("\n" + "=" * 80)
print("EJEMPLO DE CITA ANTES Y DESPUÉS DEL RESCATE PROBABILÍSTICO:")
print("=" * 80)
ejemplos_rescatados = df_clas[df_clas['etiqueta_rescatada'] == True]
if len(ejemplos_rescatados) > 0:
    ejemplo = ejemplos_rescatados.iloc[0]
    scores_ej = json.loads(ejemplo.get('scores_json', '{}')) if isinstance(ejemplo.get('scores_json'), str) else {}

    print(f"• Contexto de la cita : \"{ejemplo.get('citation_context', '')}\"")
    print(f"• Sección retórica    : {ejemplo.get('rhetorical_section', 'Unknown')}")
    if pd.notna(ejemplo.get('cited_paper_title')):
        print(f"• Artículo citado     : {ejemplo.get('cited_paper_title')}")
    print("\n[EVALUACIÓN DE SCORES ARROJADOS POR LLAMA]")
    for cat_crit in CLASES_CRITICAS:
        print(f"  - Score {cat_crit:<24}: {scores_ej.get(cat_crit, 0.0):.2f}")

    print("\n[RESULTADO DE LA DECISIÓN]")
    print(f"  - Etiqueta ANTES del rescate   : {ejemplo['predicted_label']}  (Clase saturada)")
    print(f"  - Etiqueta DESPUÉS del rescate : {ejemplo['label']}  (Clase deficitaria rescatada)")
    print(f"  - Criterio de activación       : {ejemplo.get('motivo_rescate', '')}")
else:
    print("No se encontraron instancias rescatadas en el lote actual.")
print("=" * 80)
print(f"[Guardado] Checkpoint actualizado en: {checkpoint_clasificacion}")

## 5. Submuestreo Inteligente con Priorizacion de Evidencia Documental

### Explicacion de la fase:
El objetivo de este proyecto es que el modelo no clasifique citas en el vacio, sino que pueda asociarlas con el contenido real del articulo citado (*abstract*, premisa o titulo).

Por esta razon, al momento de balancear las clases:
1. Evaluamos si la fila cuenta con texto documental autentico (`tiene_evidencia = True`).
2. Para las categorias con exceso de citas (como *Background* o *Basis* que tienen mas de 10.000), seleccionamos exactamente 2.000 instancias **tomando primero al 100% las que tienen evidencia documental verificable**.
3. Para las categorias que tienen menos de 2.000 instancias (como *Modification_Improvement* o *Further_Reading*), **conservamos el 100% de las citas autenticas disponibles**, sin descartar datos reales ni inventar textos sinteticos.


In [ ]:
# 5.1 Evaluacion de presencia de evidencia documental real
def tiene_abstract(v):
    return pd.notna(v) and len(str(v).strip()) > 25 and str(v).strip().lower() != 'nan'

def tiene_titulo(v):
    return pd.notna(v) and len(str(v).strip()) > 5 and str(v).strip().lower() != 'nan'

def tiene_premisa(v):
    return pd.notna(v) and len(str(v).strip()) > 15 and str(v).strip().lower() != 'nan'

m_abs = df_clas['cited_paper_abstract'].apply(tiene_abstract)
m_tit = df_clas['cited_paper_title'].apply(tiene_titulo)
m_pre = df_clas['sentence2'].apply(tiene_premisa) if 'sentence2' in df_clas.columns else pd.Series(False, index=df_clas.index)

df_clas['tiene_evidencia'] = m_abs | m_tit | m_pre

# 5.2 Muestreo controlado con tope de 2.000 citas por clase
SEED = 42
MAX_INSTANCIAS_POR_CLASE = 2000
subconjuntos = []

for cat, grupo in df_clas.groupby('label'):
    tot = len(grupo)
    grupo_con_ev = grupo[grupo['tiene_evidencia']].copy()
    grupo_sin_ev = grupo[~grupo['tiene_evidencia']].copy()
    n_con = len(grupo_con_ev)

    if tot > MAX_INSTANCIAS_POR_CLASE:
        # Categoria superavitaria: seleccionar 2.000 priorizando evidencia
        if n_con >= MAX_INSTANCIAS_POR_CLASE:
            seleccion = grupo_con_ev.sample(n=MAX_INSTANCIAS_POR_CLASE, random_state=SEED)
        else:
            faltan = MAX_INSTANCIAS_POR_CLASE - n_con
            seleccion = pd.concat([grupo_con_ev, grupo_sin_ev.sample(n=faltan, random_state=SEED)])
    else:
        # Categoria menor a 2.000: conservar el 100% de las citas autenticas
        seleccion = grupo.copy()

    subconjuntos.append(seleccion)

df_balanceado = pd.concat(subconjuntos, axis=0).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("=" * 90)
print(f"DATASET BALANCEADO CONFORMADO: {len(df_balanceado):,} filas")
print(f"Citas con soporte documental directo: {df_balanceado['tiene_evidencia'].sum():,} ({df_balanceado['tiene_evidencia'].mean()*100:.2f}%)")
print("=" * 90)


DATASET BALANCEADO CONFORMADO: 14,937 filas
Citas con soporte documental directo: 11,924 (79.83%)


## 6. Aislamiento Documental Estricto (*No-Leakage GroupShuffleSplit*)

### Explicacion de la fase:
¿Por que no podemos usar una particion aleatoria tradicional (`train_test_split`)?
En investigacion cientifica, un mismo articulo citante contiene decenas de citas escritas por los mismos autores, con el mismo estilo literario, vocabulario y formato. Si unas citas de ese articulo caen en Entrenamiento y otras en Prueba, el modelo puede "hacer trampa" memorizando el estilo del articulo en lugar de aprender la funcion retorica genuina (*fuga de datos o data leakage*).

Para evitar esto de forma estricta:
1. Agrupamos por el documento que cita (`citing_paper_id`).
2. Utilizamos `GroupShuffleSplit` para que el 100% de las citas de cualquier articulo queden en **una sola particion**.
3. De esta manera, garantizamos matematicamente que el solapamiento de documentos citantes, pares de citacion y textos exactos entre Entrenamiento (70%), Validacion (15%) y Prueba (15%) sea **estrictamente cero**.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# 6.1 Identificador compuesto pair_id y agrupador por documento citante
def construir_pair_id(row):
    cid = str(row.get('citing_paper_id', '')).strip()
    ref = str(row.get('cited_paper_id', '')).strip()
    if cid and ref and cid.lower() != 'nan' and ref.lower() != 'nan':
        return f"{cid}__{ref}"
    return f"single_doc_pair_{row.name}"

df_balanceado['pair_id'] = df_balanceado.apply(construir_pair_id, axis=1)

def obtener_grupo_citante(df):
    grupos = []
    for idx, row in df.iterrows():
        cid = str(row.get('citing_paper_id', '')).strip()
        if cid and cid.lower() not in ['', 'nan', 'none']:
            grupos.append(cid)
        else:
            grupos.append(f"single_doc_{idx}")
    return pd.Series(grupos, index=df.index)

grupos_doc = obtener_grupo_citante(df_balanceado)

# 6.2 Division 1: Test Humano (15%) vs Resto (85%)
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

gss_test = GroupShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=SEED)
train_val_idx, test_idx = next(gss_test.split(df_balanceado, groups=grupos_doc))

train_val_df = df_balanceado.iloc[train_val_idx].copy().reset_index(drop=True)
test_df = df_balanceado.iloc[test_idx].copy().reset_index(drop=True)

# 6.3 Division 2: Train (70%) vs Val (15%) sobre el 85% restante
grupos_train_val = obtener_grupo_citante(train_val_df)
val_relativo = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)

gss_val = GroupShuffleSplit(n_splits=1, test_size=val_relativo, random_state=SEED)
train_idx, val_idx = next(gss_val.split(train_val_df, groups=grupos_train_val))

train_df = train_val_df.iloc[train_idx].copy().reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].copy().reset_index(drop=True)

print("PARTICIONES ESTABLECIDAS CON EXITO:")
print(f"  - Train Partition      : {len(train_df):,} filas ({len(train_df)/len(df_balanceado)*100:.1f}%)")
print(f"  - Validation Partition : {len(val_df):,} filas ({len(val_df)/len(df_balanceado)*100:.1f}%)")
print(f"  - Test Humano Partition: {len(test_df):,} filas ({len(test_df)/len(df_balanceado)*100:.1f}%)")


PARTICIONES ESTABLECIDAS CON EXITO:
  - Train Partition      : 10,416 filas (69.7%)
  - Validation Partition : 2,280 filas (15.3%)
  - Test Humano Partition: 2,241 filas (15.0%)


## 7. Recuperacion Densa de Pasajes (*Top-3 Chunks*)

### Explicacion de la fase:
La propuesta del proyecto exige que para cada cita se identifiquen los 3 fragmentos (*chunks*) mas relevantes del articulo citado, con el fin de asistir al investigador en entender *por que* se hizo esa referencia.

Reglas cientificas estrictas aplicadas aqui:
1. **Segmentacion del texto citado:** Tomamos el abstract real (o premisa de SciTail, o titulo del paper) del trabajo citado y lo dividimos en oraciones independientes.
2. **Similitud semantica con Sentence-Transformers:** Comparamos el vector denso del contexto de cita (`citation_context`) contra cada oracion del trabajo citado utilizando el codificador cientifico `all-MiniLM-L6-v2`.
3. **Cero autoreferencias circulares:** Si un articulo citado no tiene abstract ni titulo registrado en Semantic Scholar (por estar bloqueado bajo paywall comercial), sus chunks se dejan estrictamente en blanco (`NaN`). Nunca usamos la propia cita como si fuera el articulo citado.


In [ ]:
from sentence_transformers import SentenceTransformer, util

# 7.1 Cargar codificador denso
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
encoder_chunks = SentenceTransformer('all-MiniLM-L6-v2', device=dispositivo)
print(f"[Chunks] Codificador cargado en: {dispositivo.upper()}")

# 7.2 Segmentador oracional limpio
def segmentar_oraciones(t):
    if pd.isna(t) or not str(t).strip() or str(t).strip().lower() == 'nan':
        return []
    norm = re.sub(r'([a-z])\.([A-Z])', r'\1. \2', str(t).strip())
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', norm) if len(s.strip()) > 12]
    vistos = set()
    unicas = []
    for s in sents:
        if s.lower() not in vistos:
            vistos.add(s.lower())
            unicas.append(s)
    return unicas

# 7.3 Pipeline de enriquecimiento de chunks
def enriquecer_particion_chunks(df_split, desc):
    df_res = df_split.copy().reset_index(drop=True)

    for k in range(1, 4):
        df_res[f'top{k}_cited_chunk'] = np.nan
        df_res[f'top{k}_section'] = np.nan
        df_res[f'top{k}_similarity'] = np.nan
        df_res[f'top{k}_position_char'] = np.nan

    print(f"\n[Extraccion] Procesando pasajes para '{desc}' ({len(df_res):,} filas)...")

    for idx in tqdm(range(len(df_res)), desc=desc):
        val_abs = df_res.at[idx, 'cited_paper_abstract'] if 'cited_paper_abstract' in df_res.columns else None
        val_pre = df_res.at[idx, 'sentence2'] if 'sentence2' in df_res.columns else None
        val_tit = df_res.at[idx, 'cited_paper_title'] if 'cited_paper_title' in df_res.columns else None

        txt_fuente = ""
        seccion_origen = ""

        if pd.notna(val_abs) and len(str(val_abs).strip()) > 25 and str(val_abs).strip().lower() != 'nan':
            txt_fuente = str(val_abs).strip()
            seccion_origen = "Abstract"
        elif pd.notna(val_pre) and len(str(val_pre).strip()) > 15 and str(val_pre).strip().lower() != 'nan':
            txt_fuente = str(val_pre).strip()
            seccion_origen = "Hypothesis_Premise"
        elif pd.notna(val_tit) and len(str(val_tit).strip()) > 5 and str(val_tit).strip().lower() != 'nan':
            txt_fuente = str(val_tit).strip()
            seccion_origen = "Title"
        else:
            # Sin soporte documental autentico: queda NaN sin inventar datos
            continue

        candidatos = segmentar_oraciones(txt_fuente)
        if not candidatos:
            candidatos = [txt_fuente]

        cit_ctx = str(df_res.at[idx, 'citation_context']).strip()
        if not cit_ctx or cit_ctx.lower() == 'nan':
            continue

        if len(candidatos) > 1:
            eq = encoder_chunks.encode(cit_ctx, convert_to_tensor=True, show_progress_bar=False)
            ec = encoder_chunks.encode(candidatos, convert_to_tensor=True, show_progress_bar=False)
            sims = util.cos_sim(eq, ec)[0].cpu().numpy()
            orden = np.argsort(-sims)
        else:
            eq = encoder_chunks.encode(cit_ctx, convert_to_tensor=True, show_progress_bar=False)
            ec = encoder_chunks.encode(candidatos[0], convert_to_tensor=True, show_progress_bar=False)
            sim = float(util.cos_sim(eq, ec)[0][0].cpu().numpy())
            orden = [0]
            sims = [sim]

        for k in range(1, 4):
            if k <= len(orden):
                c_idx = orden[k - 1]
                chunk_txt = candidatos[c_idx]
                if not chunk_txt.endswith(('.', '!', '?')):
                    chunk_txt += '.'
                df_res.at[idx, f'top{k}_cited_chunk'] = chunk_txt
                df_res.at[idx, f'top{k}_section'] = seccion_origen
                df_res.at[idx, f'top{k}_similarity'] = round(float(sims[c_idx]), 4)
                df_res.at[idx, f'top{k}_position_char'] = max(0, txt_fuente.find(candidatos[c_idx]))

    return df_res

# 7.4 Ejecucion de enriquecimiento o herencia de chunks precalculados
archivo_train = os.path.join(DRIVE_DIR, "train_enriquecido_top3_chunks.csv")
archivo_val   = os.path.join(DRIVE_DIR, "val_enriquecido_top3_chunks.csv")
archivo_test  = os.path.join(DRIVE_DIR, "test_enriquecido_top3_chunks.csv")

previo_train = "/content/drive/MyDrive/Citas_NLP/train_enriquecido_top3_chunks.csv"
previo_val   = "/content/drive/MyDrive/Citas_NLP/val_enriquecido_top3_chunks.csv"
previo_test  = "/content/drive/MyDrive/Citas_NLP/test_enriquecido_top3_chunks.csv"

heredar_chunks = False
if not os.path.exists(archivo_train) and os.path.exists(previo_train):
    try:
        df_tr_prev = pd.read_csv(previo_train)
        if df_tr_prev['label'].nunique() >= 9:
            print(f"[Optimizacion] Reutilizando particiones con Top-3 Chunks desde Citas_NLP ({len(df_tr_prev):,} filas en Train)...")
            shutil.copy2(previo_train, archivo_train)
            shutil.copy2(previo_val, archivo_val)
            shutil.copy2(previo_test, archivo_test)
            train_enriquecido = df_tr_prev
            val_enriquecido = pd.read_csv(archivo_val)
            test_enriquecido = pd.read_csv(archivo_test)
            heredar_chunks = True
    except Exception as e:
        print(f"[Aviso] No se pudieron copiar los chunks previos: {e}")

if not heredar_chunks:
    train_enriquecido = enriquecer_particion_chunks(train_df, "Train")
    val_enriquecido   = enriquecer_particion_chunks(val_df, "Validation")
    test_enriquecido  = enriquecer_particion_chunks(test_df, "Test Humano")

df_definitivo_total = pd.concat([train_enriquecido, val_enriquecido, test_enriquecido], ignore_index=True)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Chunks] Codificador cargado en: CUDA

[Extraccion] Procesando pasajes para 'Train' (10,416 filas)...


Train:   0%|          | 0/10416 [00:00<?, ?it/s]

/tmp/ipykernel_581/3424934851.py:81: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The option to remove kidney stones other than surgery is ultrasound treatments.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_res.at[idx, f'top{k}_cited_chunk'] = chunk_txt
/tmp/ipykernel_581/3424934851.py:82: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Hypothesis_Premise' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_res.at[idx, f'top{k}_section'] = seccion_origen
/tmp/ipykernel_581/3424934851.py:81: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Solely due to our extremely deep representations, we obtain a 28% relative improvement on the COCO object detection da


[Extraccion] Procesando pasajes para 'Validation' (2,280 filas)...


Validation:   0%|          | 0/2280 [00:00<?, ?it/s]

/tmp/ipykernel_581/3424934851.py:81: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'However, during this accelerated ML development process, said developers may often make sub-optimal design and implementation decisions, leading to the introduction of technical debt that, if not addressed promptly, can significantly impact on the quality of ML-based software.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_res.at[idx, f'top{k}_cited_chunk'] = chunk_txt
/tmp/ipykernel_581/3424934851.py:82: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Abstract' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_res.at[idx, f'top{k}_section'] = seccion_origen
/tmp/ipykernel_581/3424934851.py:81: FutureWarning: Setting an item of incompatible dtype is dep


[Extraccion] Procesando pasajes para 'Test Humano' (2,241 filas)...


Test Humano:   0%|          | 0/2241 [00:00<?, ?it/s]

/tmp/ipykernel_581/3424934851.py:81: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Vision-and-Language Pre-training (VLP) has improved performance on various joint vision-and-language downstream tasks.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_res.at[idx, f'top{k}_cited_chunk'] = chunk_txt
/tmp/ipykernel_581/3424934851.py:82: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Abstract' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_res.at[idx, f'top{k}_section'] = seccion_origen
/tmp/ipykernel_581/3424934851.py:81: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'In this paper, we present a minimal VLP model, Vision-and-Language Transformer (ViLT), mo

## 8. Depuracion del Esquema (23 Columnas Finales) y Persistencia en Drive

### Explicacion de la fase:
Para que los archivos sean completamente compatibles con los scripts de entrenamiento del repositorio y no contengan columnas redundantes:
* Eliminamos columnas intermedias del pipeline (`etiqueta_original`, `fuente_unificada`) que ya no son necesarias.
* Confirmamos que el archivo posea exactamente las **23 columnas esenciales** acordadas.
* Escribimos las 4 particiones oficiales directamente en la carpeta `/content/drive/MyDrive/Dataset_final`.


In [ ]:
# 8.1 Limpieza de columnas temporales redundantes
columnas_a_remover = ['etiqueta_original', 'fuente_unificada']

for col in columnas_a_remover:
    if col in df_definitivo_total.columns:
        df_definitivo_total.drop(columns=[col], inplace=True)
    if col in train_enriquecido.columns:
        train_enriquecido.drop(columns=[col], inplace=True)
    if col in val_enriquecido.columns:
        val_enriquecido.drop(columns=[col], inplace=True)
    if col in test_enriquecido.columns:
        test_enriquecido.drop(columns=[col], inplace=True)

# 8.2 Guardado en Google Drive
ruta_consolidada = os.path.join(DRIVE_DIR, "dataset_definitivo_total_chunks.csv")
ruta_train       = os.path.join(DRIVE_DIR, "train_enriquecido_top3_chunks.csv")
ruta_val         = os.path.join(DRIVE_DIR, "val_enriquecido_top3_chunks.csv")
ruta_test        = os.path.join(DRIVE_DIR, "test_enriquecido_top3_chunks.csv")

df_definitivo_total.to_csv(ruta_consolidada, index=False)
train_enriquecido.to_csv(ruta_train, index=False)
val_enriquecido.to_csv(ruta_val, index=False)
test_enriquecido.to_csv(ruta_test, index=False)

print("[Guardado] Archivos finales guardados exitosamente en Google Drive:")
print(f"  1. Maestro Consolidado : {ruta_consolidada} ({len(df_definitivo_total):,} filas)")
print(f"  2. Train (70%)         : {ruta_train} ({len(train_enriquecido):,} filas)")
print(f"  3. Validation (15%)    : {ruta_val} ({len(val_enriquecido):,} filas)")
print(f"  4. Test Humano (15%)   : {ruta_test} ({len(test_enriquecido):,} filas)")

print(f"\nEsquema de columnas confirmado ({len(df_definitivo_total.columns)} columnas):")
for i, col in enumerate(df_definitivo_total.columns, 1):
    print(f"  {i:2d}. {col}")


[Guardado] Archivos finales guardados exitosamente en Google Drive:
  1. Maestro Consolidado : /content/drive/MyDrive/Dataset_final/dataset_definitivo_total_chunks.csv (14,937 filas)
  2. Train (70%)         : /content/drive/MyDrive/Dataset_final/train_enriquecido_top3_chunks.csv (10,416 filas)
  3. Validation (15%)    : /content/drive/MyDrive/Dataset_final/val_enriquecido_top3_chunks.csv (2,280 filas)
  4. Test Humano (15%)   : /content/drive/MyDrive/Dataset_final/test_enriquecido_top3_chunks.csv (2,241 filas)

Esquema de columnas confirmado (23 columnas):
   1. citation_context
   2. rhetorical_section
   3. citing_paper_id
   4. cited_paper_id
   5. source_dataset
   6. sentence2
   7. cited_paper_title
   8. cited_paper_abstract
   9. label
  10. tiene_evidencia
  11. pair_id
  12. top1_cited_chunk
  13. top1_section
  14. top1_similarity
  15. top1_position_char
  16. top2_cited_chunk
  17. top2_section
  18. top2_similarity
  19. top2_position_char
  20. top3_cited_chunk
  21. to

## 9. Auditoria Final Integral

### Explicacion de la fase:
Presentamos las cuatro tablas de control academico y de ingenieria para verificar que el conjunto de datos cumple al 100% con los requerimientos:
1. **Distribucion por Clase:** Balance entre las 9 funciones retoricas en Train, Val y Test.
2. **Cobertura de Chunks:** Porcentaje de citas que tienen Top-1, Top-2 y Top-3 fragmentos y su similitud semantica promedio.
3. **Distribucion de Fuentes:** Proporcion de registros provenientes de SciTail, SciCite, Semantic Scholar y ACL-ARC.
4. **Verificacion de No-Leakage:** Certificacion de 0 pares documentales, 0 articulos citantes y 0 textos compartidos entre particiones.


In [ ]:
# 9.1 Tabla de Distribucion por Clase
clases_unicas = sorted(df_definitivo_total['label'].unique())
filas_clases = []
tot_global = len(df_definitivo_total)

for cat in clases_unicas:
    n_tr = (train_enriquecido['label'] == cat).sum()
    n_va = (val_enriquecido['label'] == cat).sum()
    n_te = (test_enriquecido['label'] == cat).sum()
    n_tot = (df_definitivo_total['label'] == cat).sum()

    filas_clases.append({
        'Categoria (Label)': cat,
        'Train (70%)': f"{n_tr:,}",
        'Val (15%)': f"{n_va:,}",
        'Test (15%)': f"{n_te:,}",
        'Total Filas': f"{n_tot:,}",
        'Proporcion Global': f"{(n_tot/tot_global*100):.2f}%"
    })

filas_clases.append({
    'Categoria (Label)': '=== TOTAL CORPUS ===',
    'Train (70%)': f"{len(train_enriquecido):,}",
    'Val (15%)': f"{len(val_enriquecido):,}",
    'Test (15%)': f"{len(test_enriquecido):,}",
    'Total Filas': f"{tot_global:,}",
    'Proporcion Global': '100.00%'
})

print("=" * 105)
print("1. DISTRIBUCION FINAL POR CLASE RETORICA")
print("=" * 105)
display(pd.DataFrame(filas_clases))

# 9.2 Tabla de Cobertura de Chunks
resumen_chunks = []
for nom, df_p in [("Train", train_enriquecido), ("Validation", val_enriquecido), ("Test Humano", test_enriquecido), ("Total Consolidado", df_definitivo_total)]:
    n_p = len(df_p)
    c1 = df_p['top1_cited_chunk'].notna().sum()
    c2 = df_p['top2_cited_chunk'].notna().sum()
    c3 = df_p['top3_cited_chunk'].notna().sum()
    sim = df_p['top1_similarity'].dropna().mean()
    resumen_chunks.append({
        'Particion': nom,
        'Total Filas': f"{n_p:,}",
        'Con Top-1 Chunk': f"{c1:,} ({c1/n_p*100:.1f}%)",
        'Con Top-2 Chunk': f"{c2:,} ({c2/n_p*100:.1f}%)",
        'Con Top-3 Chunks': f"{c3:,} ({c3/n_p*100:.1f}%)",
        'Similitud Cos Media': f"{sim:.4f}" if pd.notna(sim) else "N/A"
    })

print("\n" + "=" * 105)
print("2. COBERTURA DE TOP-3 CHUNKS")
print("=" * 105)
display(pd.DataFrame(resumen_chunks))

# 9.3 Tabla de Auditoria de Aislamiento Documental (No-Leakage)
def asegurar_pair_id(df):
    return df['citing_paper_id'].astype(str) + "__" + df['cited_paper_id'].astype(str)

p_tr = set(asegurar_pair_id(train_enriquecido))
p_va = set(asegurar_pair_id(val_enriquecido))
p_te = set(asegurar_pair_id(test_enriquecido))

c_tr = set(train_enriquecido['citing_paper_id'].dropna().astype(str))
c_va = set(val_enriquecido['citing_paper_id'].dropna().astype(str))
c_te = set(test_enriquecido['citing_paper_id'].dropna().astype(str))

ctx_tr = set(train_enriquecido['citation_context'].dropna().astype(str).str.strip().str.lower())
ctx_va = set(val_enriquecido['citation_context'].dropna().astype(str).str.strip().str.lower())
ctx_te = set(test_enriquecido['citation_context'].dropna().astype(str).str.strip().str.lower())

resumen_aislamiento = [
    {
        "Nivel de Aislamiento": "1. Pares Documentales (pair_id)",
        "Train <-> Val": len(p_tr.intersection(p_va)),
        "Train <-> Test": len(p_tr.intersection(p_te)),
        "Val <-> Test": len(p_va.intersection(p_te)),
        "Estado": "[OK] Sin Fuga (0 pares compartidos)"
    },
    {
        "Nivel de Aislamiento": "2. Documentos Citantes (citing_paper_id)",
        "Train <-> Val": len(c_tr.intersection(c_va)),
        "Train <-> Test": len(c_tr.intersection(c_te)),
        "Val <-> Test": len(c_va.intersection(c_te)),
        "Estado": "[OK] Sin Fuga (0 documentos compartidos)"
    },
    {
        "Nivel de Aislamiento": "3. Contexto Textual Exacto (citation_context)",
        "Train <-> Val": len(ctx_tr.intersection(ctx_va)),
        "Train <-> Test": len(ctx_tr.intersection(ctx_te)),
        "Val <-> Test": len(ctx_va.intersection(ctx_te)),
        "Estado": "[OK] Sin Duplicados (0 textos repetidos)"
    }
]

print("\n" + "=" * 105)
print("3. AUDITORIA FORMAL DE AISLAMIENTO DOCUMENTAL (NO-LEAKAGE)")
print("=" * 105)
display(pd.DataFrame(resumen_aislamiento))
print("\n[FIN] PIPELINE LINEAL CONCLUIDO CON EXITO.")


1. DISTRIBUCION FINAL POR CLASE RETORICA


,Categoria (Label),Train (70%),Val (15%),Test (15%),Total Filas,Proporcion Global
0,Application,"1,425",282,293,"2,000",13.39%
1,Background,"1,386",308,306,"2,000",13.39%
2,Basis,"1,407",294,299,"2,000",13.39%
3,Comparison,"1,388",294,301,"1,983",13.28%
4,Evidence,"1,168",291,269,"1,728",11.57%
5,Further_Reading,373,97,96,566,3.79%
6,Gap,"1,424",292,284,"2,000",13.39%
7,Identification_of_the_Originator,"1,399",322,279,"2,000",13.39%
8,Modification_Improvement,446,100,114,660,4.42%
9,=== TOTAL CORPUS ===,"10,416","2,280","2,241","14,937",100.00%



2. COBERTURA DE TOP-3 CHUNKS


,Particion,Total Filas,Con Top-1 Chunk,Con Top-2 Chunk,Con Top-3 Chunks,Similitud Cos Media
0,Train,"10,416","8,350 (80.2%)","2,849 (27.4%)","2,827 (27.1%)",0.5535
1,Validation,"2,280","1,784 (78.2%)",579 (25.4%),576 (25.3%),0.5537
2,Test Humano,"2,241","1,790 (79.9%)",633 (28.2%),621 (27.7%),0.5574
3,Total Consolidado,"14,937","11,924 (79.8%)","4,061 (27.2%)","4,024 (26.9%)",0.5541



3. AUDITORIA FORMAL DE AISLAMIENTO DOCUMENTAL (NO-LEAKAGE)


,Nivel de Aislamiento,Train <-> Val,Train <-> Test,Val <-> Test,Estado
0,1. Pares Documentales (pair_id),0,0,0,[OK] Sin Fuga (0 pares compartidos)
1,2. Documentos Citantes (citing_paper_id),0,0,0,[OK] Sin Fuga (0 documentos compartidos)
2,3. Contexto Textual Exacto (citation_context),0,0,0,[OK] Sin Duplicados (0 textos repetidos)



[FIN] PIPELINE LINEAL CONCLUIDO CON EXITO.


### Auditoría de Certeza sobre el Conjunto de Prueba
En esta celda se evalúa el nivel de certeza de las predicciones del modelo sobre el conjunto de prueba utilizando los puntajes de afinidad (`scores_json`):
1. **Cálculo de Certeza:** Se extrae la probabilidad asignada a la categoría predicha y se compara contra el umbral establecido (`THRESHOLD_CERTEZA = 0.70`).
2. **Clasificación del Estado:**
   * **Alta Certeza ($\ge 0.70$):** Instancias donde el modelo presenta un puntaje sólido y sin dispersión.
   * **Duda ($< 0.70$):** Citas con puntajes compartidos o baja confianza que ameritan inspección prioritaria.
3. **Estructura de Salida:** La columna `human_label` se inicializa vacía (`""`) sin preasignaciones automáticas, exportando el archivo `test_clasificado_humano.csv`.


In [ ]:
# ====================================================================================================
# EVALUACIÓN DE CERTEZA BASADA EN SCORES Y PREPARACIÓN DE COLUMNAS DE ANOTACIÓN
# ====================================================================================================
import os
import json
import pandas as pd
# 1. Configuración de parámetros y rutas
THRESHOLD_CERTEZA = 0.70
ruta_test = os.path.join(DRIVE_DIR, "test_enriquecido_top3_chunks.csv") if 'DRIVE_DIR' in globals() else "test_enriquecido_top3_chunks.csv"
ruta_salida = os.path.join(DRIVE_DIR, "test_clasificado_humano.csv") if 'DRIVE_DIR' in globals() else "test_clasificado_humano.csv"
# 2. Carga del conjunto de prueba
df_test = pd.read_csv(ruta_test)
print(f"[Carga] Conjunto de prueba cargado: {len(df_test):,} registros.")
# 3. Función para determinar la certeza
def evaluar_certeza_modelo(row):
    label_asignada = str(row.get('label', '')).strip()
    scores_raw = row.get('scores_json', '{}')

    scores_dict = {}
    if isinstance(scores_raw, str) and scores_raw.strip():
        try:
            scores_dict = json.loads(scores_raw)
        except Exception:
            scores_dict = {}
    elif isinstance(scores_raw, dict):
        scores_dict = scores_raw
    # Extraer la probabilidad de la clase predicha
    prob = float(scores_dict.get(label_asignada, 0.0))
    if prob == 0.0 and scores_dict:
        try:
            prob = max([float(v) for v in scores_dict.values()])
        except Exception:
            prob = 0.0
    # Determinar nivel de certeza
    certeza = "Alta Certeza" if prob >= THRESHOLD_CERTEZA else "Duda"
    human_label = ""  # Permanece vacía para la revisión
    return pd.Series([round(prob, 4), certeza, human_label])
# 4. Asignación de columnas
df_test[['score_confianza', 'certeza_evaluacion', 'human_label']] = df_test.apply(
    evaluar_certeza_modelo,
    axis=1
)
# 5. Métricas de distribución
n_alta = (df_test['certeza_evaluacion'] == "Alta Certeza").sum()
n_duda = (df_test['certeza_evaluacion'] == "Duda").sum()
total = len(df_test)
print("=" * 70)
print(f"DISTRIBUCIÓN DE CERTEZA EN EL TEST SET (Umbral = {THRESHOLD_CERTEZA:.2f})")
print("=" * 70)
print(f"• Total instancias evaluadas : {total:,}")
print(f"• Alta Certeza               : {n_alta:,} ({n_alta / total * 100:.2f}%)")
print(f"• Casos con Duda             : {n_duda:,} ({n_duda / total * 100:.2f}%)")
# 6. Guardado del archivo para el proceso de anotación
df_test.to_csv(ruta_salida, index=False)
print(f"\n[Guardado] Archivo generado exitosamente en: {ruta_salida}")